# RAG Pipeline with GradioUI
Pfizer Externship

Setting Up Colab

In [ ]:
!pip install -q gradio
!pip install -q transformers torch accelerate einops bitsandbytes
!pip install -q sentence-transformers faiss-cpu
!pip install -q pypdf pillow pytesseract pdf2image
!apt-get install -q tesseract-ocr poppler-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 335.6/335.6 kB 7.0 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 1s (238 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12)

## Step 1: Configuration
Below we will initialize a few constants and variables going forward.

In [ ]:
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "microsoft/phi-2"
CHUNK_SIZE = 300 # chars - kept small for phi's context window
CHUNK_OVERLAP = 50
TOP_K = 5 #Not too small but no too large for accuracy and speed
MAX_NEW_TOKENS = 400
PHI2_REP_PENALTY = 1.2
MIN_CHUNK_LEN = 40 # discard short chunks

## Step 2: Import List

In [ ]:
import os, re, time, math, textwrap
import gradio as gr
import numpy as np

## Step 3: Document Processing

In [ ]:
def is_scanned_pdf(path):
  try:
    import pypdf
    reader = pypdf.PdfReader(path)
    total_text = ""
    for page in reader.pages[:3]:
      total_text += page.extract_text() or ""
    return len(total_text.strip()) < 100
  except Exception:
    return False

def extract_text_digital(path):
  import pypdf
  reader = pypdf.PdfReader(path)
  pages = []
  for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    if text.strip():
      pages.append({"page": i + 1, "text": text})
  return pages

def extract_text_ocr(path):
  try:
    from pdf2image import convert_from_path
    import pytesseract
    images = convert_from_path(path, dpi=200)
    pages = []
    for i, img in enumerate(images):
      text = pytesseract.image_to_string(img)
      if text.strip():
        pages.append({"page": i + 1, "text": text})
    return pages
  except ImportError:
    return []

def load_pdf(path):
  filename = os.path.basename(path)
  if is_scanned_pdf(path):
    print(f" [{filename}] Scanned PDF detected - running OCR")
    pages = extract_text_ocr(path)
    doc_type = "scanned"
  else:
    pages = extract_text_digital(path)
    doc_type = "digital"

  if not pages:
    import pypdf
    reader = pypdf.PdfReader(path)
    pages = [{"page": i+1, "text": page.extract_text() or ""}
             for i, page in enumerate(reader.pages)]
    doc_type = "digital-fallback"

  print(f" [{filename}] {doc_type} | {len(pages)} pages extracted")
  return pages, doc_type

## Step 4: Chunking With Metadata Tagging

In [ ]:
def chunk_pages(pages, source_name, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
  chunks = []
  chunk_idx = 0
  for page_info in pages:
    page_num = page_info["page"]
    text = page_info["text"].strip()
    text = re.sub(r'\s+', ' ', text)

    if len(text) < MIN_CHUNK_LEN:
      continue

    start = 0
    while start < len(text):
      end = start + chunk_size
      chunk = text[start:end].strip()
      if len(chunk) >= MIN_CHUNK_LEN:
        chunks.append({
            "chunk_id": chunk_idx,
            "source" : source_name,
            "page" : page_num,
            "text" : chunk,
            "char_start": start,
            "char_end" : end,
        })
        chunk_idx += 1
      start += chunk_size - overlap
  print(f" [{source_name}] Chunks: {len(chunks)}")
  return chunks

## Step 5: Embedding and FAISS Index

In [ ]:
_embed_model = None

def get_embed_model():
  global _embed_model
  if _embed_model is None:
    from sentence_transformers import SentenceTransformer
    print(f" Loading embedding model: {EMBED_MODEL_NAME}")
    _embed_model = SentenceTransformer(EMBED_MODEL_NAME)
  return _embed_model

def embed_texts(texts):
  model = get_embed_model()
  return model.encode(texts, show_progress_bar=False, convert_to_numpy=True)

def build_faiss_index(chunks):
  import faiss
  texts = [c["text"] for c in chunks]
  embeddings = embed_texts(texts).astype("float32")
  dim = embeddings.shape[1]
  index = faiss.IndexFlatIP(dim) # inner product = cosine on normalized vectors
  faiss.normalize_L2(embeddings)
  index.add(embeddings)
  print(f" FAISS index build - {index.ntotal} vectors, dim={dim}")
  return index, embeddings

def retrieve_chunks(query, index, chunks, top_k=TOP_K):
  import faiss
  q_vec = embed_texts([query]).astype("float32")
  faiss.normalize_L2(q_vec)
  scores, indices = index.search(q_vec, top_k)

  results = []
  for score, idx in zip(scores[0], indices[0]):
    if idx < 0:
      continue
    chunk = chunks[idx].copy()
    chunk["confidence"] = float(score)
    results.append(chunk)
  return results

## Step 6: PHI-2 LLM

In [ ]:
_llm_model = None
_llm_tokenizer = None


def load_phi2():
  global _llm_model, _llm_tokenizer
  if _llm_model is not None:
    return _llm_model, _llm_tokenizer

  import torch
  from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

  print(f" Loading LLM: {LLM_MODEL_NAME}")
  tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME, trust_remote_code=True)
  if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

  config = AutoConfig.from_pretrained(LLM_MODEL_NAME, trust_remote_code=True)
  config.pad_token_id = tokenizer.pad_token_id

  try:
    from transformers import BitsAndBytesConfig
    bnb = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_use_double_quant = True,
        bnb_4bit_quant_type = "nf4",
        bnb_4bit_compute_dtype = torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME, config=config, quantization_config=bnb, device_map="auto", trust_remote_code=True
    )
    print(" Phi-2 loaded in 4-bit (BitsAndBytes)")
  except Exception:
    model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME, config=config, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True,
    )
    print(" Phi-2 Loaded in float16")

  model.resize_token_embeddings(len(tokenizer))
  model.eval()
  _llm_model, _llm_tokenizer = model, tokenizer
  return model, tokenizer

def build_prompt(query, retrieved_chunks):
  context_parts = []
  for i, chunk in enumerate(retrieved_chunks, 1):
    context_parts.append(
        f"[{i}] Source: {chunk['source']}, Page {chunk['page']}\n{chunk['text']}"
    )
  context = "\n\n".join(context_parts)

  return (
      "Instruct: You are a precise document assistant."
      "Answer the question using ONLY the provided context."
      "After each fact, cite the source in brackets like [Source, Page N]."
      "If the answer is not in the context, say 'This information is not in the uploaded documents.'\n\n"
      f"Context:\n{context}\n\n"
      f"Question: {query}\n"
      "Output:"
  )

def clean_phi2_output(raw, prompt):
  if prompt in raw:
    answer = raw[raw.index(prompt) + len(prompt):]
  else:
    answer = re.split(r'\bOutput\s*:', raw, maxsplit=1)[-1]
  answer = answer.strip()

  seen, clean = set(), []
  for line in answer.split("\n"):
    key = line.strip().lower()
    if key and key not in seen:
      seen.add(key)
      clean.append(line)
  answer = "\n".join(clean).strip()
  if len(answer) > 2000:
    answer = answer[:2000].rsplit(".", 1)[0] + "."
  return answer

def generate_answer(query, retrieved_chunks):
  import torch
  model, tokenizer = load_phi2()
  prompt = build_prompt(query, retrieved_chunks)
  inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2000).to(model.device)
  with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        repetition_penalty=PHI2_REP_PENALTY,
        pad_token_id = tokenizer.pad_token_id,
        eos_token_id = tokenizer.eos_token_id,
    )
  raw = tokenizer.decode(output_ids[0], skip_special_tokens=True)
  return clean_phi2_output(raw, prompt)

## Step 7: Handle Global State

In [ ]:
state = {
    "chunks" : [],
    "index" : None,
    "doc_names": [],
}

def reset_state():
  state["chunks"] = []
  state["index"] = None
  state["doc_names"] = []

## Step 8: Gradio Handlers

In [ ]:
import traceback

def process_documents(files, progress=gr.Progress()):
  if not files:
    return "No files uploaded.", gr.update(interactive=False)
  try:
    reset_state()
    all_chunks = []
    progress(0, desc="Starting...")

    for i, file in enumerate(files):
      name = os.path.basename(file.name)
      progress((i / len(files)) * 0.6, desc=f"Processing {name}...")
      pages, _ = load_pdf(file.name)
      chunks = chunk_pages(pages, source_name=name)
      all_chunks.extend(chunks)
      state["doc_names"].append(name)

    if not all_chunks:
      return "No text could be extracted from the uploaded files.", gr.update(interactive=False)

    progress(0.7, desc="Building embeddings...")
    index, _ = build_faiss_index(all_chunks)

    state["chunks"] = all_chunks
    state["index"] = index

    progress(1.0, desc="DONE!")

    docs_list = "\n".join(f" {n}" for n in state["doc_names"])
    status = (
        f" Ready! Indexed {len(all_chunks)} chunks from"
        f" {len(state['doc_names'])} document(s):\n{docs_list}"
    )
    return status, gr.update(interactive=True)
  except Exception:
    err = traceback.format_exc()
    print(err)
    return f" Error: {err}", gr.update(interactive=False)

def chat(message, history):
  if state["index"] is None:
    reply = "Please upload at least one document first."
    history.append((message, reply))
    return history, ""

  try:
    t0 = time.time()

    #Retrieve
    hits = retrieve_chunks(message, state["index"], state["chunks"], top_k=TOP_K)

    #Generate
    answer = generate_answer(message, hits)
    elapsed = time.time() - t0

    #Build source footnote
    seen_sources = {}
    for h in hits:
      key = f"{h['source']} p.{h['page']}"
      seen_sources[key] = max(seen_sources.get(key, 0), h["confidence"])

    source_lines = "\n".join(
        f" [{k}] confidence: {v:2f}"
        for k, v in sorted(seen_sources.items(), key=lambda x: -x[1])
    )

    full_reply = (
        f"{answer}\n\n"
        f"---\n"
        f"Sources ({len(hits)} chunks retrieved in {elapsed:.1f}s):\n"
        f"{source_lines}"
    )

    history.append((message, full_reply))
    return history, ""
  except Exception:
    err = traceback.format_exc()
    print(err)
    history.append((message, f"Error:\n{err}"))
    return history, ""

def clear_chat():
  return [], ""

## Step 9: GradioUI

In [ ]:
CSS = """
/* ── Base & Typography ─────────────────────────────────────── */
@import url('https://fonts.googleapis.com/css2?family=DM+Serif+Display&family=DM+Sans:wght@300;400;500&family=JetBrains+Mono:wght@400;500&display=swap');

body, .gradio-container {
    font-family: 'DM Sans', sans-serif !important;
    background: #0f1117 !important;
    color: #e8e6e0 !important;
}

/* ── Header ────────────────────────────────────────────────── */
.header-block {
    background: linear-gradient(135deg, #1a1f2e 0%, #0f1117 100%);
    border: 1px solid #2a3040;
    border-radius: 12px;
    padding: 28px 32px 20px;
    margin-bottom: 20px;
}
.header-title {
    font-family: 'DM Serif Display', serif !important;
    font-size: 2rem !important;
    color: #f0ece3 !important;
    margin: 0 0 4px 0 !important;
    letter-spacing: -0.5px;
}
.header-sub {
    font-size: 0.85rem !important;
    color: #6b7280 !important;
    font-weight: 300;
    margin: 0 !important;
}
.pill {
    display: inline-block;
    background: #1e2d40;
    color: #60a5fa;
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.72rem;
    padding: 2px 10px;
    border-radius: 99px;
    border: 1px solid #2a4060;
    margin-right: 6px;
    margin-top: 10px;
}

/* ── Upload Panel ──────────────────────────────────────────── */
.upload-panel {
    background: #151821 !important;
    border: 1px solid #252b3a !important;
    border-radius: 10px !important;
    padding: 16px !important;
}
.upload-panel label {
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.8rem !important;
    color: #9ca3af !important;
    text-transform: uppercase;
    letter-spacing: 0.08em;
}

/* ── Status box ────────────────────────────────────────────── */
.status-box textarea {
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 0.78rem !important;
    background: #0d1117 !important;
    color: #4ade80 !important;
    border: 1px solid #1f2d1f !important;
    border-radius: 8px !important;
}

/* ── Chat window ───────────────────────────────────────────── */
.chatbot-window {
    background: #111318 !important;
    border: 1px solid #252b3a !important;
    border-radius: 10px !important;
}
.chatbot-window .message.user {
    background: #1e2d40 !important;
    color: #bfdbfe !important;
    border-radius: 8px 8px 2px 8px !important;
    font-size: 0.9rem !important;
}
.chatbot-window .message.bot {
    background: #181c27 !important;
    color: #e2e8f0 !important;
    border-radius: 2px 8px 8px 8px !important;
    font-size: 0.9rem !important;
    border-left: 3px solid #3b82f6 !important;
    white-space: pre-wrap !important;
    font-family: 'DM Sans', sans-serif !important;
}

/* ── Input row ─────────────────────────────────────────────── */
.chat-input textarea {
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.9rem !important;
    background: #151821 !important;
    color: #e2e8f0 !important;
    border: 1px solid #2a3040 !important;
    border-radius: 8px !important;
}
.chat-input textarea:focus {
    border-color: #3b82f6 !important;
    box-shadow: 0 0 0 2px rgba(59,130,246,0.15) !important;
}

/* ── Buttons ───────────────────────────────────────────────── */
.btn-primary {
    background: #2563eb !important;
    color: #fff !important;
    border: none !important;
    border-radius: 8px !important;
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 500 !important;
    font-size: 0.85rem !important;
    transition: background 0.15s !important;
}
.btn-primary:hover { background: #1d4ed8 !important; }
.btn-secondary {
    background: #1e2535 !important;
    color: #9ca3af !important;
    border: 1px solid #2a3040 !important;
    border-radius: 8px !important;
    font-family: 'DM Sans', sans-serif !important;
    font-size: 0.82rem !important;
}
.btn-secondary:hover { border-color: #4b5563 !important; color: #d1d5db !important; }

/* ── Suggested questions ───────────────────────────────────── */
.suggested-label {
    font-size: 0.75rem !important;
    color: #6b7280 !important;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    margin-bottom: 6px !important;
}
.suggest-btn {
    background: #151821 !important;
    color: #93c5fd !important;
    border: 1px solid #1e2d40 !important;
    border-radius: 6px !important;
    font-size: 0.78rem !important;
    font-family: 'JetBrains Mono', monospace !important;
    text-align: left !important;
    padding: 6px 10px !important;
}
.suggest-btn:hover { background: #1e2d40 !important; border-color: #3b82f6 !important; }

/* ── Side info cards ────────────────────────────────────────── */
.info-card {
    background: #13161f;
    border: 1px solid #252b3a;
    border-radius: 8px;
    padding: 14px 16px;
    margin-bottom: 12px;
}
.info-card-title {
    font-size: 0.72rem;
    color: #6b7280;
    text-transform: uppercase;
    letter-spacing: 0.1em;
    margin-bottom: 6px;
}
.info-card-val {
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.8rem;
    color: #93c5fd;
}
"""

SUGGESTED = [
    "What are the storage conditions for this product?",
    "What test methods were used for quality control?",
    "Summarize the key specifications in this document.",
    "What sterilization method was used?",
    "What is the shelf life of this product?",
]

def build_ui():
  with gr.Blocks(css=CSS, title="RAG Chatbot") as demo:
    gr.HTML("""
    <div class="header-block">
      <p class="header-title">RAG Document Assistant</p>
      <p class="header-sub">Upload PDFs -> Ask questions -> Get grounded answers with citations</p>
      <span class="pill">Phi-2</span>
      <span class="pill">sentence-transfomrers</span>
      <span class="pill">FAISS</span>
      <span class="pill">OCR-ready</span>
    </div>
    """)
    with gr.Row():
      with gr.Column(scale=1, min_width=200):
        gr.HTML('<div class="info-card-title" style="margin-bottom:8ox">Documents</div>')
        file_input = gr.File(
            label="Upload PDF(s)",
            file_types=[".pdf"],
            file_count="multiple",
            elem_classes=["upload-panel"],
        )
        process_btn = gr.Button(
            "Process Documents",
            elem_classes=["btn-primary"],
        )

        status_box = gr.Textbox(
            label="Status",
            lines=4,
            interactive=False,
            elem_classes=["status-box"],
            value="Upload PDF files above, then click Process.",
        )

        gr.HTML('<hr style="border-color:#252b3a;margin:16px 0">')
        gr.HTML('<div class="suggested-label"> Try asking</div>')

        for q in SUGGESTED:
          gr.HTML(f'<div class="suggest-btn" style="margin-bottom:5px">{q}</div>')
          gr.HTML("""
          <div class="info-card">
            <div class="info-card-val">
              MiniLM-L6 -> FAISS<br>
              top_k = 5 chunks<br>
              Phi-2 (4-bit)
            </div>
          </div>
          <div class="info-card">
            <div class="info-card-title">Chunk settings</div>
            <div class="info-card-val">
              size = 300 chars<br>
              overlap = 50 chars<br>
              OCR auto-detected
            </div>
          </div>
          """)
      with gr.Column(scale=3):

        chatbot = gr.Chatbot(
            label="",
            height=480,
            elem_classes=["chatbot-window"],
            show_label=False,
            bubble_full_width=False,

        )

        with gr.Row():
          msg_input = gr.Textbox(
              placeholder="Ask a question about your documents...",
              show_label=False,
              lines=1,
              scale=5,
              interactive=False,
              elem_classes=["chat-input"],
          )

          send_btn = gr.Button(
              "Send ->",
              scale=1,
              interactive=False,
              elem_classes=["btn-primary"],
          )

        with gr.Row():
          clear_btn = gr.Button(
              "Clear",
              elem_classes=["btn-secondary"],
          )
    process_btn.click(
        fn=process_documents,
        inputs=[file_input],
        outputs=[status_box, msg_input],
    ).then(
        fn=lambda: gr.update(interactive=True),
        outputs=[send_btn],
    )

    send_btn.click(
        fn=chat,
        inputs=[msg_input,chatbot],
        outputs=[chatbot,msg_input],
    )

    msg_input.submit(
        fn=chat,
        inputs=[msg_input,chatbot],
        outputs=[chatbot, msg_input],
    )

    clear_btn.click(
        fn=clear_chat,
        outputs=[chatbot, msg_input],
    )
  return demo

In [ ]:
if __name__ == "__main__":
  print("\n" + "="*60)
  print(" RAG Chatbot - Starting up")
  print(" Loading embedding model on first query...")
  print(" Loading Phi-2 on first query (one-time ~2 min)...")
  print("=" * 60 + "\n")

  demo = build_ui()
  demo.launch(share=True, debug=False)


 RAG Chatbot - Starting up
 Loading embedding model on first query...
 Loading Phi-2 on first query (one-time ~2 min)...



/tmp/ipykernel_1584/511968315.py:179: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="RAG Chatbot") as demo:
/tmp/ipykernel_1584/511968315.py:236: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1584/511968315.py:236: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1584/511968315.py:236: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7db2dc0eedf537cbf6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 10: Testing and Deployment
This was all with the help of Claude

In [ ]:
# -*- coding: utf-8 -*-
"""
================================================================================
  RAG PIPELINE — EVALUATION & METRICS
  Recall@K · MRR · Precision@K · End-to-End Accuracy · System Performance
================================================================================

HOW TO USE
----------
1. Run this cell AFTER your main pipeline cells have executed
   (all functions like retrieve_chunks, generate_answer, state, etc. must be live)
2. Make sure at least one PDF has been processed through process_documents()
3. Run this cell — a full evaluation report prints to the output
4. Results are also saved to evaluation_report.json for your Google Doc

WHAT IS MEASURED
----------------
  Retrieval   → Recall@K, Precision@K, MRR
  Accuracy    → Exact match, partial match, no-answer detection, source citation
  Performance → OCR time, chunking time, embedding time, retrieval latency,
                LLM generation time, total end-to-end time
"""

import time
import json
import re
import os
from datetime import datetime

# ── TEST SET ──────────────────────────────────────────────────────────────────
# Each entry defines:
#   query          — the question sent to the pipeline
#   relevant_terms — keywords that MUST appear in a correct answer (used for
#                    Recall/Precision — any chunk containing these counts as relevant)
#   expected_ans   — keywords that should appear in the generated answer
#   expect_no_ans  — True if the correct answer is "not in the document"
# ─────────────────────────────────────────────────────────────────────────────
TEST_QUERIES = [
    {
        "query"         : "What are the storage conditions for this product?",
        "relevant_terms": ["storage", "temperature", "15", "25", "sealed"],
        "expected_ans"  : ["15", "25", "sealed", "store"],
        "expect_no_ans" : False,
    },
    {
        "query"         : "What test methods were used for quality control?",
        "relevant_terms": ["titration", "karl fischer", "icp", "usp", "ftir", "ph"],
        "expected_ans"  : ["titration", "usp", "test"],
        "expect_no_ans" : False,
    },
    {
        "query"         : "Summarize the key specifications in this document.",
        "relevant_terms": ["purity", "ph", "moisture", "specification", "result"],
        "expected_ans"  : ["purity", "ph", "specification"],
        "expect_no_ans" : False,
    },
    {
        "query"         : "What sterilization method was used?",
        "relevant_terms": ["steriliz", "filtration", "0.22", "membrane", "not"],
        "expected_ans"  : ["not", "steriliz"],
        "expect_no_ans" : False,
    },
    {
        "query"         : "What is the shelf life of this product?",
        "relevant_terms": ["shelf", "month", "expir", "manufactur"],
        "expected_ans"  : ["month", "shelf"],
        "expect_no_ans" : False,
    },
    {
        "query"         : "Who is the CEO of the manufacturing company?",
        "relevant_terms": ["ceo", "chief", "executive", "officer"],
        "expected_ans"  : ["not in", "cannot", "no information"],
        "expect_no_ans" : True,  # should refuse rather than hallucinate
    },
]

K = TOP_K  # use the same K as the pipeline

# ── HELPERS ───────────────────────────────────────────────────────────────────

def chunk_is_relevant(chunk_text, relevant_terms):
    """A chunk is relevant if it contains at least one of the relevant terms."""
    text_lower = chunk_text.lower()
    return any(term.lower() in text_lower for term in relevant_terms)

def answer_is_correct(answer_text, expected_keywords, expect_no_ans):
    """
    Partial-match accuracy: answer is correct if it contains at least half
    of the expected keywords (case-insensitive).
    For no-answer queries, correct means the answer signals it doesn't know.
    """
    ans_lower = answer_text.lower()
    if expect_no_ans:
        no_ans_signals = ["not in", "cannot", "no information",
                          "not available", "not found", "not mentioned",
                          "not in the uploaded"]
        return any(s in ans_lower for s in no_ans_signals)
    matches = sum(1 for kw in expected_keywords if kw.lower() in ans_lower)
    return matches >= max(1, len(expected_keywords) // 2)

def has_citation(answer_text):
    """Check whether the answer contains a source citation like [filename, Page N]."""
    return bool(re.search(r'\[.+[Pp]age\s*\d+.*\]', answer_text))

def measure_time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs) and return (result, elapsed_seconds)."""
    t0 = time.time()
    result = fn(*args, **kwargs)
    return result, time.time() - t0

# ── PERFORMANCE BASELINE: measure pipeline stages independently ───────────────

def measure_pipeline_stages(pdf_path=None):
    """
    Re-runs each pipeline stage individually to get clean per-stage timings.
    Uses the first doc_name from state if no path is provided.
    """
    timings = {}

    # If a real PDF is available, re-measure OCR + chunking
    if pdf_path and os.path.exists(pdf_path):
        _, ocr_time   = measure_time(load_pdf, pdf_path)
        timings["ocr_and_extraction_s"] = round(ocr_time, 3)

        pages, _ = load_pdf(pdf_path)
        chunks_result, chunk_time = measure_time(
            chunk_pages, pages, os.path.basename(pdf_path)
        )
        timings["chunking_s"] = round(chunk_time, 3)

        _, embed_time = measure_time(build_faiss_index, chunks_result)
        timings["embedding_and_indexing_s"] = round(embed_time, 3)
    else:
        timings["ocr_and_extraction_s"] = "n/a (no PDF path provided)"
        timings["chunking_s"]           = "n/a"
        timings["embedding_and_indexing_s"] = "n/a"

    return timings

# ── MAIN EVALUATION LOOP ──────────────────────────────────────────────────────

def run_evaluation():
    print("\n" + "=" * 64)
    print("  RAG PIPELINE EVALUATION")
    print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  {len(TEST_QUERIES)} queries  |  K = {K}")
    print("=" * 64)

    if state["index"] is None or not state["chunks"]:
        print("\n  ERROR: No documents indexed. Upload and process a PDF first.")
        return

    # ── Per-query results ─────────────────────────────────────────────────────
    query_results   = []
    retrieval_times = []
    generation_times= []
    e2e_times       = []

    recall_at_k     = []
    precision_at_k  = []
    mrr_scores      = []
    accuracy_scores = []
    citation_scores = []

    for qi, test in enumerate(TEST_QUERIES, 1):
        query         = test["query"]
        rel_terms     = test["relevant_terms"]
        exp_ans       = test["expected_ans"]
        expect_no_ans = test["expect_no_ans"]

        print(f"\n  Query {qi}/{len(TEST_QUERIES)}: {query}")

        # ── Retrieval ─────────────────────────────────────────────────────────
        hits, ret_time = measure_time(
            retrieve_chunks, query, state["index"], state["chunks"], top_k=K
        )
        retrieval_times.append(ret_time)

        # Relevance flags for each retrieved chunk
        relevance_flags = [
            chunk_is_relevant(h["text"], rel_terms) for h in hits
        ]
        n_relevant_retrieved = sum(relevance_flags)

        # Recall@K — did we retrieve at least one relevant chunk?
        rec_k = 1.0 if n_relevant_retrieved > 0 else 0.0
        recall_at_k.append(rec_k)

        # Precision@K — what fraction of retrieved chunks are relevant?
        prec_k = n_relevant_retrieved / len(hits) if hits else 0.0
        precision_at_k.append(prec_k)

        # MRR — reciprocal rank of the first relevant chunk
        rr = 0.0
        for rank, flag in enumerate(relevance_flags, 1):
            if flag:
                rr = 1.0 / rank
                break
        mrr_scores.append(rr)

        # Confidence scores
        top_conf   = round(hits[0]["confidence"], 4) if hits else 0.0
        mean_conf  = round(sum(h["confidence"] for h in hits) / len(hits), 4) if hits else 0.0

        print(f"    Retrieval  : {ret_time*1000:.0f} ms  |  "
              f"Relevant chunks: {n_relevant_retrieved}/{len(hits)}  |  "
              f"Top conf: {top_conf:.3f}")

        # ── Generation ────────────────────────────────────────────────────────
        answer, gen_time = measure_time(generate_answer, query, hits)
        generation_times.append(gen_time)

        e2e = ret_time + gen_time
        e2e_times.append(e2e)

        # ── Accuracy ──────────────────────────────────────────────────────────
        correct  = answer_is_correct(answer, exp_ans, expect_no_ans)
        cited    = has_citation(answer)
        accuracy_scores.append(1.0 if correct else 0.0)
        citation_scores.append(1.0 if cited    else 0.0)

        print(f"    Generation : {gen_time:.1f}s  |  "
              f"Correct: {'✓' if correct else '✗'}  |  "
              f"Cited: {'✓' if cited else '✗'}")
        print(f"    Answer preview: {answer[:120].strip()}...")

        query_results.append({
            "query"             : query,
            "recall_at_k"       : rec_k,
            "precision_at_k"    : prec_k,
            "mrr"               : round(rr, 4),
            "top_confidence"    : top_conf,
            "mean_confidence"   : mean_conf,
            "relevant_retrieved": n_relevant_retrieved,
            "total_retrieved"   : len(hits),
            "answer_correct"    : correct,
            "has_citation"      : cited,
            "retrieval_ms"      : round(ret_time * 1000, 1),
            "generation_s"      : round(gen_time, 2),
            "e2e_s"             : round(e2e, 2),
            "answer"            : answer,
        })

    # ── Aggregate metrics ─────────────────────────────────────────────────────
    n = len(TEST_QUERIES)

    avg_recall    = sum(recall_at_k)    / n
    avg_precision = sum(precision_at_k) / n
    avg_mrr       = sum(mrr_scores)     / n
    avg_accuracy  = sum(accuracy_scores)/ n
    avg_citation  = sum(citation_scores)/ n

    avg_ret_ms  = sum(retrieval_times)  / n * 1000
    avg_gen_s   = sum(generation_times) / n
    avg_e2e_s   = sum(e2e_times)        / n
    total_e2e_s = sum(e2e_times)

    # ── Print summary ─────────────────────────────────────────────────────────
    div = "=" * 64
    print(f"\n{div}")
    print("  RETRIEVAL METRICS")
    print(div)
    print(f"  Recall@{K}    : {avg_recall:.3f}  "
          f"({'✓ good' if avg_recall >= 0.7 else '✗ low — consider larger K or better chunking'})")
    print(f"  Precision@{K} : {avg_precision:.3f}  "
          f"({'✓ good' if avg_precision >= 0.5 else '✗ low — chunks may be too broad'})")
    print(f"  MRR          : {avg_mrr:.3f}  "
          f"({'✓ good' if avg_mrr >= 0.6 else '✗ low — relevant chunk not ranking first'})")

    print(f"\n{div}")
    print("  END-TO-END ACCURACY")
    print(div)
    print(f"  Answer accuracy  : {avg_accuracy:.1%}  "
          f"({sum(accuracy_scores):.0f}/{n} correct)")
    print(f"  Citation rate    : {avg_citation:.1%}  "
          f"({sum(citation_scores):.0f}/{n} answers cited sources)")

    print(f"\n{div}")
    print("  SYSTEM PERFORMANCE")
    print(div)
    print(f"  Avg retrieval latency : {avg_ret_ms:.1f} ms")
    print(f"  Avg LLM generation    : {avg_gen_s:.1f} s")
    print(f"  Avg end-to-end        : {avg_e2e_s:.1f} s")
    print(f"  Total eval time       : {total_e2e_s:.1f} s  "
          f"({n} queries)")

    print(f"\n{div}")
    print("  PER-QUERY BREAKDOWN")
    print(div)
    header = f"  {'Query':<42} {'R@K':>4} {'P@K':>4} {'MRR':>5} {'Acc':>4} {'Cite':>5} {'E2E':>6}"
    print(header)
    print("  " + "-" * 62)
    for i, (test, res) in enumerate(zip(TEST_QUERIES, query_results), 1):
        short_q = test["query"][:40] + ".." if len(test["query"]) > 40 else test["query"]
        print(f"  {short_q:<42} "
              f"{res['recall_at_k']:>4.2f} "
              f"{res['precision_at_k']:>4.2f} "
              f"{res['mrr']:>5.3f} "
              f"{'✓' if res['answer_correct'] else '✗':>4} "
              f"{'✓' if res['has_citation'] else '✗':>5} "
              f"{res['e2e_s']:>5.1f}s")
    print(div)

    # ── Save JSON report ──────────────────────────────────────────────────────
    report = {
        "timestamp"       : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "k"               : K,
        "n_queries"       : n,
        "aggregate": {
            "recall_at_k"       : round(avg_recall,    4),
            "precision_at_k"    : round(avg_precision, 4),
            "mrr"               : round(avg_mrr,       4),
            "answer_accuracy"   : round(avg_accuracy,  4),
            "citation_rate"     : round(avg_citation,  4),
            "avg_retrieval_ms"  : round(avg_ret_ms,    2),
            "avg_generation_s"  : round(avg_gen_s,     2),
            "avg_e2e_s"         : round(avg_e2e_s,     2),
            "total_eval_s"      : round(total_e2e_s,   2),
        },
        "per_query": query_results,
    }

    with open("evaluation_report.json", "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  Full report saved → evaluation_report.json")
    print(div + "\n")
    return report

# ── RUN ───────────────────────────────────────────────────────────────────────
report = run_evaluation()


  RAG PIPELINE EVALUATION
  2026-04-15 01:59:44
  6 queries  |  K = 5

  ERROR: No documents indexed. Upload and process a PDF first.
